# Implementation of sleep scoring from Watson et al., Neuron (2016)
The paper uses three data sources to classify arousal stages into Awake/NREM/REM/Microarousal, and we want to attempt a further classification of Awake into Active and Quite Wakefulness.

Three data sources are:
- Broadband LFP
- Narrowband theta
- Intracranial EMG

All analyses were originally carried out on ephys recordings from neocortex, downsampled to 1250Hz.

**Broadband LFP** (Low-frequency spectral features)
1. Short-time FFT: 10s window, 1s step, evaluated at log-spaced frequencies between 1–100 Hz
2. z-score
3. PCA
4. Pick PC1. If negative loading on <20Hz frequencies, flip sign.
5. Pick trough in bimodal distribution as threshold between NREM and Wake/REM.

**Narrowband theta**
- Ratio of 5-10Hz/2-16Hz power. Hippocampal channels used where present.

**Intracranial EMG**
1. Bandpass (Butterworth, 4th order) filter in 300-600Hz range
2. Subselect pairs of random, good channels ≥ 2 shanks apart
3. Take mean of Pearson's R in 500ms windows --> **this is the EMG score**

### Imports

In [ ]:
import os
import yaml
import warnings

import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt
import spikeinterface.full as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as sp

from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from scipy.stats import gaussian_kde
from diptest import diptest
from probeinterface import read_probeinterface
from dotenv import load_dotenv, find_dotenv
from pathlib import Path

import sleep_sandbox.analysis as sa
from sleep_sandbox.io import (
    find_amplifier_files, remote_available, get_or_build, load_preprocessed,
    load_bno055, concat_bno055, align_bno055_to_lfp,
)
from sleep_sandbox.preprocessing import build_band, imu_kinematics
from sleep_sandbox.analysis import (
    log_spectrogram, broadband_pc1, select_channel_by_dip, pc1_threshold_gmm,
    smooth_norm, bimodal_thresh, band_power, theta_ratio,
    select_theta_channel, select_theta_channel_peak, conditioned_theta_thresh, zscore, bin_max,
)


### Load configs and set up recording object

In [ ]:
load_dotenv()
repo_root = Path(find_dotenv()).parent

remote_data_dir = os.getenv("REMOTE_DATA_DIR")
derivatives_dir = os.getenv("DERIVATIVES_DIR")
local_parent = os.getenv("LOCAL_DERIV_PARENT")

# Sleep-scoring algorithm parameters (channel selection, thresholds, theta bands, EMG, IMU).
with open(repo_root / "config/sleep_scoring.yml") as f:
    scoring_config = yaml.safe_load(f)

# Theta band conventions are a module-level default in sleep_sandbox.analysis; override them from
# config so select_theta_channel / theta_ratio (which look them up internally by name) stay in
# sync with the bands defined in config/sleep_scoring.yml.
sa.THETA_BANDS = {
    name: {"theta": tuple(band["theta"]), "denom": tuple(band["denom"])}
    for name, band in scoring_config["theta"]["conventions"].items()
}

# Recording selection: which probe + which AmplifierData _N segment(s) to concatenate.
probe_name = "ProbeB"
suffixes = None                              # int / range / list of AmplifierData _N segments

# Streams the pipeline needs; local mode loads these, remote mode builds them.
streams = ["lfp_nocmr", "lfp_cmr", "emg"]

# Local-first: use local derivatives if all present, even when the remote is mounted.
local_parent = Path(local_parent) if local_parent else None
use_local = local_parent is not None and all(any(local_parent.glob(f"{s}__*")) for s in streams)

if use_local:
    print(f"Local derivatives found; loading from {local_parent}")
elif remote_available(remote_data_dir):
    warnings.warn("No complete set of local derivatives; building from remote raw data.")
else:
    raise RuntimeError(
        "No local derivatives and REMOTE_DATA_DIR not available. "
        "Set LOCAL_DERIV_PARENT to a folder of prebuilt derivatives, or mount the remote raw data."
    )

In [ ]:
# Remote build path: discover raw segments, load config, concatenate. Skipped when use_local.
if not use_local:
    session_dir = Path(remote_data_dir).parent   # datetime dir holding probe config + NeuropixelsV2/

    try:
        amplifier_paths, probe_config_path = find_amplifier_files(session_dir, probe_name, suffixes)
    except FileNotFoundError:
        probe_config_path = Path(os.getenv("PROBE_CONFIG_PATH"))

    # Concatenation is only valid for a contiguous run of segments (no gaps).
    seg_ns = sorted(int(p.stem.split("_")[-1]) for p in amplifier_paths)
    assert seg_ns == list(range(seg_ns[0], seg_ns[-1] + 1)), f"non-contiguous segments: {seg_ns}"
    seg_source = f"{probe_name}_seg{seg_ns[0]}-{seg_ns[-1]}"

    # Load preprocessing config
    config_path = os.path.join(repo_root, "config/preprocessing.yml")
    with open(config_path) as f:
        config = yaml.safe_load(f)

    sample_rate = config['recording']['sample_rate']
    n_channels = config['recording']['n_channels']
    dtype = config['recording']['dtype']
    freq_min = config['lfp']['bandpass_filter']['freq_min']
    freq_max = config['lfp']['bandpass_filter']['freq_max']
    freq_min_highpass = config['preprocessing']['bandpass_filter']['freq_min']
    freq_max_highpass = config['preprocessing']['bandpass_filter']['freq_max']
    reference = config['lfp']['common_reference']['reference']
    operator = config['lfp']['common_reference']['operator']
    resample_rate = config['lfp']['resample_rate']
    margin_ms = config['lfp']['margin_ms']

    emg_freq_min = config['emg']['bandpass_filter']['freq_min']
    emg_freq_max = config['emg']['bandpass_filter']['freq_max']
    emg_margin_ms = config['emg']['margin_ms']
    emg_resample_rate = config['emg']['resample_rate']

    # Concatenate the selected segments into one continuous recording (bridges internal boundaries).
    segments = [se.read_binary(p, sample_rate, dtype, n_channels) for p in amplifier_paths]
    recording = si.concatenate_recordings(segments)

### Add probe

In [ ]:
if not use_local:
    probe_group = read_probeinterface(probe_config_path)
    probe = probe_group.probes[0]

    active_mask = probe.device_channel_indices != -1
    active_probe = probe.get_slice(active_mask)

    # active_probe has 384 contacts, all mapped 1:1 to recording channels.
    # The full probe has 5120 contacts (384 active); copy_metadata in preprocessors silently
    # drops it when contact count != channel count, causing the "no Probe attached" warning.
    recording_with_probe = recording.set_probe(active_probe)

### Preprocess

We want to filter this in two ways for a first pass:
- Broadband LFP, roughly 0.5-300 Hz
- Intracranial EMG band, 300-600 Hz

In [ ]:
# Preprocessing chains (lazy - no data loaded yet). build_band returns each recording together
# with its canonical step list, so the chain and the params keying its derivative can't desync.
# margin_ms must be long enough for the 0.25 Hz high-pass corner (settles over seconds); LFP and
# EMG each use their own margin from config/preprocessing.yml (EMG's 300-600 Hz band settles fast).
if not use_local:
    # Shared input provenance: the exact ordered concatenated segments, pinned into every hash.
    concat_step = ("concatenate", {"inputs": [p.stem for p in amplifier_paths]})

    # LFP band ± CMR, and EMG band
    recording_lfp_no_cmr, lfp_nocmr_steps = build_band(
        recording_with_probe, freq_min, freq_max, resample_rate, margin_ms, source_steps=[concat_step])
    recording_lfp, lfp_cmr_steps = build_band(
        recording_with_probe, freq_min, freq_max, resample_rate,  margin_ms,
        cmr=True, reference=reference, operator=operator, source_steps=[concat_step])
    recording_emg, emg_steps = build_band(
        recording_with_probe, emg_freq_min, emg_freq_max, emg_resample_rate, emg_margin_ms, source_steps=[concat_step])

In [ ]:
if not use_local:
    preprocessed_dir = Path(derivatives_dir) / "preprocessed"

    # Each call loads from disk if the param-matched folder exists, else builds + saves it.
    # steps come straight from build_band above, so they always mirror the actual chain.
    recording_lfp_no_cmr = get_or_build(recording_lfp_no_cmr, seg_source, lfp_nocmr_steps, preprocessed_dir, "lfp_nocmr", n_jobs=4, chunk_duration="30s")
    recording_lfp = get_or_build(recording_lfp, seg_source, lfp_cmr_steps, preprocessed_dir, "lfp_cmr", n_jobs=20, chunk_duration="30s")
    recording_emg = get_or_build(recording_emg, seg_source, emg_steps, preprocessed_dir, "emg", n_jobs=6, chunk_duration="30s")

In [ ]:
# Local load path: read prebuilt derivatives by stream name (no raw / no mount needed).
if use_local:
    recording_lfp_no_cmr = load_preprocessed(local_parent, "lfp_nocmr")
    recording_lfp = load_preprocessed(local_parent, "lfp_cmr")
    recording_emg = load_preprocessed(local_parent, "emg")

### Dominant spectral feature in broadband LFP

In [ ]:
# Slow-wave analysis functions (log_spectrogram, broadband_pc1, select_channel_by_dip,
# pc1_threshold) are imported from sleep_sandbox.analysis -- see the imports cell.


In [ ]:
# Channel selection: scan every Nth channel, score PC1 bimodality with Hartigan's dip test.
channel_stride = scoring_config["channel_selection"]["stride"]                   # scan every Nth channel
manual_channel = scoring_config["channel_selection"]["manual_slow_wave_channel"]  # None = automatic

# Shared buzcode-style spectrogram / thresholding constants, used throughout below.
pc_index = scoring_config["slow_wave"]["pc_index"]
orientation_freq_hz = scoring_config["slow_wave"]["orientation_freq_hz"]
spec_cfg = scoring_config["spectrogram"]
spectrogram_kwargs = dict(window_s=spec_cfg["window_s"], step_s=spec_cfg["step_s"],
                          freq_min=spec_cfg["freq_min"], freq_max=spec_cfg["freq_max"],
                          n_freq_bins=spec_cfg["n_freq_bins"])
step_s = spec_cfg["step_s"]                                     # also the metric bin width for smooth_norm
smooth_win_s = scoring_config["smoothing"]["window_s"]
bt_startbins = scoring_config["bimodal_threshold"]["startbins"]
bt_maxbins = scoring_config["bimodal_threshold"]["maxbins"]

rec = recording_lfp
fs = rec.get_sampling_frequency()

best_channel, dip_stats, candidate_idx = select_channel_by_dip(
    rec, fs, channel_stride, manual_channel, pc=pc_index, orientation_freq_hz=orientation_freq_hz,
    **spectrogram_kwargs)
print(f"Selected channel {best_channel} (dip={dip_stats[best_channel]:.4f})")

In [ ]:
# Bimodality (Hartigan dip) across scanned channels
plt.figure(figsize=(10, 3))
plt.plot(candidate_idx, dip_stats[candidate_idx], marker=".")
plt.axvline(best_channel, color="r", ls="--", label=f"best = {best_channel}")
plt.xlabel("channel index")
plt.ylabel("dip statistic")
plt.legend()

In [ ]:
# Run the broadband slow-wave pipeline on the selected channel.
sw_channel = rec.channel_ids[best_channel]
sw_trace = rec.get_traces(channel_ids=[sw_channel]).squeeze()
sw_pc1, sw_pc1_loading, sw_spec, sw_freqs, sw_times = broadband_pc1(
    sw_trace, fs, pc=pc_index, orientation_freq_hz=orientation_freq_hz, **spectrogram_kwargs)

# buzcode-concordant metric: 15 s-smoothed, [0,1]-normalised PC1. All thresholding/masking below
# is done on sw_metric (not raw sw_pc1) so it matches ClusterStates_GetMetrics units.
sw_metric = smooth_norm(sw_pc1, step_s=step_s, win_s=smooth_win_s)


In [ ]:
# Plot spectrogram of the selected channel
plt.figure(figsize=(10, 4))
plt.pcolormesh(sw_times, sw_freqs, np.log10(sw_spec), shading="auto", cmap="viridis")
plt.yscale("log")
plt.colorbar(label="Power")

In [ ]:
# Plot the buzcode-concordant slow-wave metric (15 s-smoothed, [0,1]) over time with its threshold.
plt.figure(figsize=(10, 3), dpi=150)
plt.plot(sw_times, sw_metric, lw=0.5)
plt.xlabel("Time (s)")
plt.ylabel("slow-wave metric (smoothed, [0,1])")
plt.title("abcGolden01 broadbandSlowWave (ch{}, with CMR)".format(sw_channel))
sw_threshold = bimodal_thresh(sw_metric, bt_startbins, bt_maxbins)   # == pc1_threshold(sw_pc1); metric units
plt.axhline(sw_threshold, color="r", ls="--", label=f"threshold = {sw_threshold:.3f}")
plt.legend()


In [ ]:
# Step 5 (buzcode-concordant): threshold = histogram trough between the two modes (bz_BimodalThresh),
# on the smoothed/normed metric. NREM candidate = metric above threshold (compare in metric units).
sw_threshold = bimodal_thresh(sw_metric, bt_startbins, bt_maxbins)
sw_nrem = sw_metric > sw_threshold


In [ ]:
# Distribution of the slow-wave metric with the between-mode trough threshold (buzcode bz_BimodalThresh).
plt.figure(figsize=(6, 4))
plt.hist(sw_metric, bins=50, density=True, alpha=0.4)
x = np.linspace(sw_metric.min(), sw_metric.max(), 1000)
plt.plot(x, gaussian_kde(sw_metric)(x), color="k", lw=1, label="KDE")
plt.axvline(sw_threshold, color="r", ls="--", label=f"threshold = {sw_threshold:.3f}")
plt.xlabel("slow-wave metric (smoothed, [0,1])")
plt.ylabel("density")
plt.legend()


### Check classification agreement between channels

Check correlation of binary classifications of sleep state between "best" channel (highest dip test score) and all other channels. Each channel's PC1 is smoothed + [0,1]-normalised, then binarised > vs ≤ the between-mode histogram trough (buzcode `bz_BimodalThresh`).

In [ ]:
# For each scanned channel: broadband PC1 -> smooth/norm -> histogram-trough threshold -> binary NREM.
# Correlate each channel's binary series against the best channel's (all in metric units).

binary_by_channel = {}
for ci in candidate_idx:
    pc1 = broadband_pc1(rec.get_traces(channel_ids=[rec.channel_ids[ci]]).squeeze(), fs,
                         pc=pc_index, orientation_freq_hz=orientation_freq_hz, **spectrogram_kwargs)[0]
    m = smooth_norm(pc1, step_s=step_s, win_s=smooth_win_s)
    binary_by_channel[ci] = (m > bimodal_thresh(m, bt_startbins, bt_maxbins)).astype(int)

ref_binary = binary_by_channel[best_channel]
corr_by_channel = {ci: np.corrcoef(b, ref_binary)[0, 1] for ci, b in binary_by_channel.items()}


In [ ]:
# Correlation vs channel, coloured by diptest quintile.
ch = np.array(sorted(corr_by_channel))
try:
    corr = np.array([corr_by_channel[c] for c in ch])
except:
    corr = np.load("/Users/willreith/Documents/phd/aeon/sleep_sandbox/notebooks/abcGolden01_binary-sleep-class_corr-with-best-channel_full-probe_channel-indexed.npy")
#dip_stats = np.load("/Users/willreith/Documents/phd/aeon/sleep_sandbox/notebooks/abcGolden01_hartigans-diptest_full-probe_channel-indexed.npy")
dips = dip_stats[ch]
quintile = np.digitize(dips, np.quantile(dips, [0.2, 0.4, 0.6, 0.8]))

plt.figure(figsize=(10, 3))
sc = plt.scatter(ch, corr, c=quintile, cmap="viridis")
plt.axvline(best_channel, color="r", ls="--", label=f"best = {best_channel}")
plt.colorbar(sc, label="diptest quintile", ticks=range(5))
plt.xlabel("channel index")
plt.ylabel("corr with best-channel binary")
plt.legend()

### Narrowband theta

In [ ]:
# Narrowband theta functions (THETA_BANDS, band_power, theta_ratio, select_theta_channel)
# are imported from sleep_sandbox.analysis -- see the imports cell.


In [ ]:
# Run every configured convention on the CMR LFP (recording_lfp). Each picks its own best theta
# channel, then we take that channel's theta-ratio series. Time base matches sw_times (10 s / 1 s).
theta_series = {}
theta_channels = {}
theta_times = None
for conv in scoring_config["theta"]["conventions"]:
    ch, dip, theta_candidates = select_theta_channel(recording_lfp, fs, conv, channel_stride, **spectrogram_kwargs)
    trace = recording_lfp.get_traces(channel_ids=[recording_lfp.channel_ids[ch]]).squeeze()
    spec, freqs, theta_times = log_spectrogram(trace, fs, **spectrogram_kwargs)
    theta_series[conv] = theta_ratio(spec, freqs, conv)
    theta_channels[conv] = ch
    print(f"{conv}: theta channel {ch} (dip={dip[ch]:.4f})")

In [ ]:
# Compare the two conventions: z-scored log-ratio over time, and against each other.
# z-score the log ratio (matches buzcode / the slow-wave metric; ratios are right-skewed).
z_watson = zscore(np.log10(theta_series["watson"]))
z_shin = zscore(np.log10(theta_series["shin"]))

fig, (ax_ts, ax_sc) = plt.subplots(1, 2, figsize=(13, 3), gridspec_kw={"width_ratios": [3, 1]})
ax_ts.plot(theta_times, z_watson, lw=0.5, label=f"watson (ch {theta_channels['watson']})")
ax_ts.plot(theta_times, z_shin, lw=0.5, label=f"shin (ch {theta_channels['shin']})")
ax_ts.set_xlabel("Time (s)")
ax_ts.set_ylabel("z-scored log theta ratio")
ax_ts.legend()

ax_sc.scatter(z_watson, z_shin, s=2, alpha=0.3)
r = np.corrcoef(z_watson, z_shin)[0, 1]
ax_sc.set_xlabel("watson")
ax_sc.set_ylabel("shin")
ax_sc.set_title(f"r = {r:.3f}")
fig.tight_layout()

In [ ]:
# Shin theta measure vs slow-wave PC1 (both z-scored for the overlay; same 10 s / 1 s time base).
z_shin = zscore(np.log10(theta_series["shin"]))
z_sw = zscore(sw_pc1)

# Correlation
plt.figure(figsize=(4, 4), dpi=150)
plt.scatter(z_sw, z_shin, s=2, alpha=0.3)
r = np.corrcoef(z_sw, z_shin)[0, 1]
plt.xlabel("slow-wave PC1 (z)")
plt.ylabel("shin theta (z)")
plt.title(f"r = {r:.3f}")

# Time series
plt.figure(figsize=(10, 3), dpi=150)
plt.plot(sw_times, z_sw, lw=0.5, label="slow-wave PC1")
plt.plot(theta_times, z_shin, lw=0.5, label=f"Shin theta (ch {theta_channels['shin']})")
plt.title("abcGolden01: slow-wave PC1 vs Shin theta (z-scored)")
plt.xlabel("Time (s)")
plt.ylabel("z-score")
plt.legend()

### IMU movement (BNO055) vs sleep metrics

Concatenate all BNO055 blocks, derive translational + angular speed and linear-acceleration
magnitude, and compare against slow-wave PC1 and the Shin narrowband-theta measure. Alignment (`align_bno055_to_lfp`): each block's IMU is placed on the LFP concat timeline via the
shared 250 MHz ONIX clock (`Bno055_Clock_N` ↔ `ProbeB_Clock_N`).

In [ ]:
# Align BNO055 to the LFP concat timeline via the shared 250 MHz ONIX clock
# (Bno055_Clock_N <-> ProbeB_Clock_N). Drops inter-block gaps + corrects the nominal-vs-true rate.
bno_dir = Path(os.getenv("RAW_DATA_DIR"))     # NeuropixelsV2 dir holding the Bno055 + ProbeB_Clock .bin files
imu_data, imu_t, imu_valid, imu_edges = align_bno055_to_lfp(bno_dir)
imu_kin_cfg = scoring_config["imu"]["kinematics"]
imu_speed, imu_ang, imu_accel = imu_kinematics(
    imu_data, imu_t, imu_edges, hp_fc=imu_kin_cfg["highpass_fc_hz"], hp_order=imu_kin_cfg["highpass_order"])

# Duration check: aligned IMU timeline vs the LFP concat timeline (should now match closely).
print(f"IMU (aligned): {imu_t[imu_valid][-1]:.1f} s  ({imu_valid.sum()} valid / {imu_valid.size} samples)")
print(f"LFP sw_times:  {sw_times[-1]:.1f} s")
print(f"mismatch:      {sw_times[-1] - imu_t[imu_valid][-1]:.1f} s")

In [ ]:
# Reduce IMU (~100 Hz) to the 1 s PC1/theta grid: max per bin (catches brief movements).
# Only bin gap-valid samples (approach-B mask), and drop samples outside the LFP extent so the
# final segment's ~53 s clock overrun (IMU mapped beyond sw_times[-1]) doesn't pollute the edge bins.
tv = imu_t[imu_valid]
imu_speed_b = bin_max(tv, imu_speed[imu_valid], sw_times)
imu_ang_b = bin_max(tv, imu_ang[imu_valid], sw_times)
imu_accel_b = bin_max(tv, imu_accel[imu_valid], sw_times)

In [ ]:
# Figure 1: PC1, Shin theta, and IMU movement over the session (shared 1 s time base).
z_shin = zscore(np.log10(theta_series["shin"]))
lower = 6200
upper = 6500

fig, ax = plt.subplots(5, 1, sharex=True, figsize=(12, 9), dpi=150)
ax[0].plot(sw_times[lower:upper], sw_pc1[lower:upper], lw=0.5)
ax[0].axhline(sw_threshold, color="r", ls="--")
ax[0].set_ylabel("PC1 (slow wave)")
ax[1].plot(sw_times[lower:upper], z_shin[lower:upper], lw=0.5, color="C1")
ax[1].set_ylabel("shin theta (z)")
ax[2].plot(sw_times[lower:upper], imu_speed_b[lower:upper], lw=0.5, color="C2")
ax[2].set_ylabel("speed (m/s)")
ax[3].plot(sw_times[lower:upper], imu_ang_b[lower:upper], lw=0.5, color="C3")
ax[3].set_ylabel("ang. speed (deg/s)")
ax[4].plot(sw_times[lower:upper], imu_accel_b[lower:upper], lw=0.5, color="C4")
ax[4].set_ylabel("|accel| (m/s²)")
ax[4].set_xlabel("Time (s)")
fig.tight_layout()

In [ ]:
# Figure 2: each IMU movement signal vs each sleep metric (NaN bins dropped; r per panel).
movement = [("speed", imu_speed_b[lower:upper]), ("ang speed", imu_ang_b[lower:upper]), ("|accel|", imu_accel_b[lower:upper])]
metrics = [("PC1", sw_pc1[lower:upper]), ("shin theta (z)", z_shin[lower:upper])]

fig, axes = plt.subplots(len(movement), len(metrics), figsize=(9, 10))
for i, (mn, mv) in enumerate(movement):
    for j, (nn, nv) in enumerate(metrics):
        a = axes[i, j]
        ok = ~np.isnan(mv)
        a.scatter(nv[ok], mv[ok], s=2, alpha=0.3)
        r = np.corrcoef(nv[ok], mv[ok])[0, 1]
        a.set_title(f"r = {r:.2f}")
        if i == len(movement) - 1:
            a.set_xlabel(nn)
        if j == 0:
            a.set_ylabel(mn)
fig.tight_layout()

### Mismatch due to IMU data overrun

IMU data overruns amplifier data by ~53s at the end. This does not affect analysis, but here are some sanity checks that pins it outside of the main recording.

In [ ]:
# (1) Reconcile the true durations instead of comparing to sw_times[-1].
fs_raw = 30000  # nominal ephys sample rate (align_bno055_to_lfp default)

lfp_dur = recording_lfp.get_total_duration()
clock_blocks = sorted(int(p.stem.split("_")[-1]) for p in bno_dir.glob("*ProbeB_Clock_*.bin"))
clock_sum = sum(len(np.memmap(bno_dir / f"NeuropixelsV2_ProbeB_Clock_{b}.bin", dtype="<u8", mode="r")) / fs_raw
                for b in clock_blocks)
imu_end = imu_t[imu_valid][-1]

print(f"LFP recording total:       {lfp_dur:8.1f} s  ({recording_lfp.get_num_frames()} frames @ {recording_lfp.get_sampling_frequency():.0f} Hz)")
print(f"ProbeB_Clock sum (n={len(clock_blocks)}):   {clock_sum:8.1f} s")
print(f"aligned IMU end (valid):   {imu_end:8.1f} s")
print(f"sw_times[-1] (window ctr): {sw_times[-1]:8.1f} s")
print("---")
print(f"LFP - clock_sum:           {lfp_dur - clock_sum:+8.1f} s   (0 => derivative built from exactly these segments)")
print(f"LFP - aligned IMU end:     {lfp_dur - imu_end:+8.1f} s   (small +ve => IMU tail ends slightly early)")
print(f"LFP - sw_times[-1]:        {lfp_dur - sw_times[-1]:+8.1f} s   (~+5 s => STFT window-centre offset)")

In [ ]:
# (2) Event-lag cross-correlation: does movement line up with the neural wake proxy?
# Wake proxy = low slow-wave PC1 (-sw_pc1). Both on the 1 s grid; peak at lag 0 => truly aligned.
speed_z = zscore(np.nan_to_num(imu_speed_b[lower:upper], nan=0.0))
wake_z = zscore(-sw_pc1[lower:upper])
n = len(speed_z)
xc = np.correlate(speed_z, wake_z, mode="full") / n
lags = np.arange(-n + 1, n)                       # bins == seconds (1 s grid)
peak = int(lags[np.argmax(xc)])

win = 120
sel = np.abs(lags) <= win
plt.figure(figsize=(8, 3))
plt.plot(lags[sel], xc[sel], lw=0.8)
plt.axvline(peak, color="r", ls="--", label=f"peak lag = {peak} s")
plt.axvline(0, color="k", lw=0.5)
plt.xlabel("lag (s): speed relative to wake proxy")
plt.ylabel("cross-correlation")
plt.legend()
print(f"peak cross-correlation at lag {peak} s (r = {xc.max():.3f})")

### Intracranial EMG

Watson et al. (2016) derive an EMG proxy from the LFP itself: volume-conducted muscle activity
correlates high-frequency (300-600 Hz) signals across spatially distant sites. High cross-shank
correlation = high muscle tone (wake); it collapses under REM atonia. This complements the
slow-wave PC1 (which flags NREM) by separating REM from quiet wake.

Recipe: mean Pearson's *r* over 500 ms windows across random channel pairs ≥ 2 shanks apart
(300-600 Hz band, already built as `recording_emg`), then aligned to the 1 s slow-wave grid.
Functions live here for now; migrate to `analysis.py` later.

In [ ]:
# EMG functions (to migrate to analysis.py later).

def make_emg_pairs(shank_ids, n_pairs=100, min_shank_dist=2, seed=0):
    """n_pairs random channel pairs whose shanks are >= min_shank_dist apart. Pairs are spread
    round-robin across every qualifying shank-combo, drawing distinct channels (a shank's pool is
    reshuffled + refilled only once exhausted) so as many electrodes as possible are covered."""
    rng = np.random.default_rng(seed)
    shanks = np.asarray(shank_ids, dtype=int)
    uniq = np.unique(shanks)
    ch_by_shank = {s: np.where(shanks == s)[0] for s in uniq}
    combos = [(a, b) for i, a in enumerate(uniq) for b in uniq[i + 1:] if b - a >= min_shank_dist]
    queues = {s: [] for s in uniq}
    def draw(s):
        if not queues[s]:
            queues[s] = list(rng.permutation(ch_by_shank[s]))
        return int(queues[s].pop())
    pairs = np.array([draw(s) for k in range(n_pairs) for s in combos[k % len(combos)]]).reshape(-1, 2)
    return pairs, combos


def emg_from_lfp(rec, pairs, win_s=0.5):
    """Watson EMG score: mean per-window Pearson r across channel pairs, on non-overlapping win_s
    windows of the raw 300-600 Hz trace. Returns (emg, times) at the window rate (1/win_s)."""
    fs = rec.get_sampling_frequency()
    win = int(round(win_s * fs))
    n_win = rec.get_num_frames() // win
    ch_ids = rec.channel_ids
    def windowed(c):  # channel index -> [n_win, win] float64 (per-pair reload; memmap slice)
        x = rec.get_traces(channel_ids=[ch_ids[c]], end_frame=n_win * win).squeeze().astype(np.float64)
        return x.reshape(n_win, win)
    r = np.zeros(n_win)
    for a, b in pairs:
        A, B = windowed(a), windowed(b)
        A = A - A.mean(1, keepdims=True)
        B = B - B.mean(1, keepdims=True)
        r += (A * B).sum(1) / np.sqrt((A ** 2).sum(1) * (B ** 2).sum(1))
    emg = r / len(pairs)
    times = (np.arange(n_win) * win + win / 2) / fs
    return emg, times


In [ ]:
# Build cross-shank pairs (>= min_shank_dist shanks apart) and compute the EMG score, then align to
# the 1 s slow-wave grid (compute at emg.window_s, interpolate onto sw_times). Params from
# config/sleep_scoring.yml (emg section).
emg_cfg = scoring_config["emg"]
emg_shanks = recording_emg.get_probes()[0].shank_ids.astype(int)
emg_pairs, emg_combos = make_emg_pairs(
    emg_shanks, n_pairs=emg_cfg["n_pairs"], min_shank_dist=emg_cfg["min_shank_dist"], seed=emg_cfg["seed"])
print(f"{len(emg_pairs)} pairs across shank-combos {emg_combos}; "
      f"distinct channels used: {len(np.unique(emg_pairs))}")

emg_score, emg_times = emg_from_lfp(recording_emg, emg_pairs, win_s=emg_cfg["window_s"])
emg_b = np.interp(sw_times, emg_times, emg_score)   # onto the 1 s slow-wave grid


In [ ]:
# EMG score vs slow-wave metric and IMU speed over the session (shared 1 s grid).
z_emg = zscore(emg_b)
fig, ax = plt.subplots(3, 1, sharex=True, figsize=(12, 6), dpi=150)
ax[0].plot(sw_times, z_emg, lw=0.4)
ax[0].set_ylabel("EMG (z)")
ax[1].plot(sw_times, sw_metric, lw=0.4, color="C0")
ax[1].axhline(sw_threshold, color="r", ls="--")
ax[1].set_ylabel("slow-wave metric [0,1]")
ax[2].plot(sw_times, imu_speed_b, lw=0.4, color="C2")
ax[2].set_ylabel("speed (m/s)")
ax[2].set_xlabel("Time (s)")
fig.suptitle("abcGolden01: intracranial EMG vs slow-wave metric and IMU speed")
fig.tight_layout()


In [ ]:
# EMG vs movement / sleep metrics (NaN IMU bins dropped; r per panel).
panels = [("IMU speed (m/s)", imu_speed_b), ("PC1 (slow wave)", sw_pc1), ("shin theta (z)", z_shin)]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), dpi=150)
for ax, (name, v) in zip(axes, panels):
    ok = ~np.isnan(v) & ~np.isnan(emg_b)
    ax.scatter(v[ok], emg_b[ok], s=2, alpha=0.3)
    r = np.corrcoef(v[ok], emg_b[ok])[0, 1]
    ax.set_title(f"r = {r:.2f}")
    ax.set_xlabel(name)
axes[0].set_ylabel("EMG score")
fig.tight_layout()


In [ ]:
# Interactive 3D trajectory through EMG / theta / PC1 state space (drag to rotate; needs ipympl).
lower=0
upper=6971
theta_measure = theta_series["watson"]  # or theta_series["watson"] for the other convention
pc1 = (sw_pc1[lower:upper]+abs(min(sw_pc1[lower:upper])))/(max(sw_pc1[lower:upper])+abs(min(sw_pc1[lower:upper])))  # normalise to [0, 1]
theta = (theta_measure[lower:upper]+abs(min(theta_measure[lower:upper])))/(max(theta_measure[lower:upper])+abs(min(theta_measure[lower:upper])))  # normalise to [0, 1]
emg = (emg_b[lower:upper]+abs(min(emg_b[lower:upper])))/(max(emg_b[lower:upper])+abs(min(emg_b[lower:upper])))  # normalise to [0, 1]
move_thresh = scoring_config["imu"]["movement_speed_threshold_mps"]
%matplotlib widget
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(pc1, theta, emg, c=imu_speed_b[lower:upper]>move_thresh, cmap="viridis", s=3, alpha=0.5)
ax.view_init(elev=20, azim=45)
ax.set_xlabel("Watson theta ratio")
ax.set_ylabel("Slow-wave PC1")
ax.set_zlabel("EMG score")
fig.colorbar(sc, label="translational speed (m/s)", shrink=0.6)
fig.tight_layout()


In [ ]:
# Same 3D state-space trajectory, coloured by time (drag to rotate; needs ipympl).
lower=0
upper=6971
theta_measure = theta_series["watson"]  # or theta_series["shin"] for the other convention
pc1 = (sw_pc1[lower:upper]+abs(min(sw_pc1[lower:upper])))/(max(sw_pc1[lower:upper])+abs(min(sw_pc1[lower:upper])))  # normalise to [0, 1]
theta = (theta_measure[lower:upper]+abs(min(theta_measure[lower:upper])))/(max(theta_measure[lower:upper])+abs(min(theta_measure[lower:upper])))  # normalise to [0, 1]
emg = (emg_b[lower:upper]+abs(min(emg_b[lower:upper])))/(max(emg_b[lower:upper])+abs(min(emg_b[lower:upper])))  # normalise to [0, 1]
%matplotlib widget
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(pc1, theta, emg, c=sw_times[lower:upper], cmap="viridis", s=3, alpha=0.5)
ax.view_init(elev=20, azim=45)
ax.set_xlabel("Slow-wave PC1")
ax.set_ylabel("Watson theta ratio")
ax.set_zlabel("EMG score")
fig.colorbar(sc, label="time (s)", shrink=0.6)
fig.tight_layout()


In [ ]:
# Revert to inline so only the 3D figures above stay widgets. Run right after the 3D cells.
# plt.close("all") discards the still-open widget figures; the widget backend never auto-closes
# them, so otherwise they pile up and all flush into the next inline plot. (This also freezes the
# 3D widgets above -- fine once you're done rotating them.)
%matplotlib inline
plt.close("all")


In [ ]:
# Distribution of the narrowband theta score.
theta_measure = theta_series["shin"]  # or theta_series["shin"] for the other convention
emg_threshold = bimodal_thresh(smooth_norm(theta_measure, step_s=step_s, win_s=smooth_win_s), bt_startbins, bt_maxbins)
plt.figure(figsize=(6, 4))
plt.hist(theta_measure, bins=50, density=True, alpha=0.6)
plt.axvline(emg_threshold, color="r", ls="--", label=f"threshold = {emg_threshold*100:.2f}")
plt.xlabel("Theta score")
plt.ylabel("density")
plt.title("abcGolden01: Theta score distribution")


In [ ]:
# Distribution of the EMG score (1 s grid).

emg_threshold = bimodal_thresh(smooth_norm(emg_b, step_s=step_s, win_s=smooth_win_s), bt_startbins, bt_maxbins)
plt.figure(figsize=(6, 4))
plt.hist(emg_b, bins=50, density=True, alpha=0.6)
plt.axvline(emg_threshold, color="r", ls="--", label=f"threshold = {emg_threshold*100:.2f}")
plt.xlabel("EMG score")
plt.ylabel("density")
plt.title("abcGolden01: EMG score distribution")


### Concordant thresholds (buzcode `SleepScoreMaster` order)

SW → motion (EMG) → movement-conditioned theta, each thresholded at its between-mode histogram
trough (`bz_BimodalThresh`) on the 15 s-smoothed, [0,1] metric. Theta channel via `peakTH`
(highest theta-power ratio, not dip). The theta trough is taken on **non-moving** epochs only
(`MOVtimes = low SW & high motion`), with a NREM-excluded fallback. Stops at thresholds — no
NREM/MOV/REM state labels yet.

In [ ]:
# Concordant thresholds in buzcode order. Theta channel via peakTH (not dip); theta ratio on it,
# smoothed+normed. Motion = EMG score, smoothed+normed. Each threshold = histogram trough; theta
# is conditioned on non-moving epochs. (Needs sw_metric, sw_times, emg_b from the cells above.)
peak_band = scoring_config["theta"]["channel_peak_band"]
concordant_conv = scoring_config["theta"]["concordant_convention"]
th_ch, th_peak, _ = select_theta_channel_peak(
    recording_lfp, fs, theta=tuple(peak_band["theta"]), denom=tuple(peak_band["denom"]), stride=channel_stride,
    **spectrogram_kwargs)
th_spec, th_freqs, _ = log_spectrogram(
    recording_lfp.get_traces(channel_ids=[recording_lfp.channel_ids[th_ch]]).squeeze(), fs, **spectrogram_kwargs)
# channel_peak_band (buzcode PickSWTHChannel) picks the channel; concordant_conv's band computes the ratio.
theta_metric = smooth_norm(theta_ratio(th_spec, th_freqs, concordant_conv), step_s=step_s, win_s=smooth_win_s)
motion_metric = smooth_norm(emg_b, step_s=step_s, win_s=smooth_win_s)

sw_thresh = bimodal_thresh(sw_metric, bt_startbins, bt_maxbins)
motion_thresh = bimodal_thresh(motion_metric, bt_startbins, bt_maxbins)
th_thresh, movtimes = conditioned_theta_thresh(
    theta_metric, sw_metric, motion_metric, sw_thresh, motion_thresh, bt_startbins, bt_maxbins)

print(f"peakTH theta channel {th_ch} (peak ratio={th_peak[th_ch]:.4f})")
print(f"swthresh={sw_thresh:.3f}  MotionThresh(EMG)={motion_thresh:.3f}  MOV frac={movtimes.mean():.3f}")
print(f"THthresh conditioned={th_thresh:.3f}   unconditioned={bimodal_thresh(theta_metric, bt_startbins, bt_maxbins):.3f}")


In [ ]:
# Effect of movement-conditioning on the theta threshold (left) and the motion (EMG) split (right).
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), dpi=150)
ax[0].hist(theta_metric, bins=50, density=True, alpha=0.4, label="all epochs")
ax[0].hist(theta_metric[~movtimes], bins=50, density=True, alpha=0.4, label="non-moving")
ax[0].axvline(th_thresh, color="r", ls="--", label=f"THthresh={th_thresh:.3f}")
ax[0].set_xlabel("theta metric [0,1]"); ax[0].set_ylabel("density"); ax[0].legend()
ax[1].hist(motion_metric, bins=50, density=True, alpha=0.5)
ax[1].axvline(motion_thresh, color="r", ls="--", label=f"MotionThresh={motion_thresh:.3f}")
ax[1].set_xlabel("motion metric (EMG) [0,1]"); ax[1].legend()
fig.tight_layout()


### IMU vs EMG as the motion signal

buzcode supports an accelerometer as a drop-in motion source (`MotionSource='Accelerometer'`,
`SleepScoreMaster.m`; `bz_getIntanAccel.m`): 3-axis magnitude (`vecnorm`) → ~1 Hz Butterworth
low-pass → `abs` → downsample to 2 Hz; `ClusterStates_GetMetrics` then uses its **variance**
(`data_var`) through the *same* smooth-15 s → norm[0,1] → histogram-trough → `MOVtimes` pipeline
as EMG. So the IMU is interchangeable with EMG downstream. Below we A/B EMG against IMU translational
speed / |accel| / angular speed through identical `smooth_norm`→`bimodal_thresh`→conditioned-theta,
comparing `MotionThresh`, `MOV frac`, and the conditioned `THthresh`.

In [ ]:
# IMU motion candidates on the SW grid. buzcode's accel path uses accelerometer VARIANCE (data_var),
# so add a per-bin variance of |accel|. Also test each metric's bimodality (Hartigan dip) -- the whole
# MOVtimes step assumes the smoothed motion is bimodal (low vs high tone / rest vs active).
# NB: EMG = muscle TONE, not movement; only tone (atonia) can isolate REM, the IMU can't.
tv = imu_t[imu_valid]
def bin_var(t, x, grid):  # per-grid-bin variance (buzcode 'data_var' analogue); NaN for empty bins
    idx = np.searchsorted(grid, t).clip(0, len(grid) - 1)
    c = np.bincount(idx, minlength=len(grid)).astype(float)
    s = np.bincount(idx, weights=x, minlength=len(grid))
    s2 = np.bincount(idx, weights=x * x, minlength=len(grid))
    c[c == 0] = np.nan
    return s2 / c - (s / c) ** 2
motion_candidates = {
    "EMG (tone)": emg_b,
    "IMU speed": np.interp(sw_times, tv, imu_speed[imu_valid]),
    "IMU |accel|": np.interp(sw_times, tv, imu_accel[imu_valid]),
    "IMU angular": np.interp(sw_times, tv, imu_ang[imu_valid]),
    "IMU accel-var": np.nan_to_num(bin_var(tv, imu_accel[imu_valid], sw_times)),  # buzcode data_var analogue
}
motion_metrics = {}
print(f"{'motion':14s} {'MotThr':>7s} {'MOVfrac':>8s} {'THcond':>7s} {'dip':>7s} {'p':>6s}")
for name, sig in motion_candidates.items():
    m = smooth_norm(sig, step_s=step_s, win_s=smooth_win_s)
    mt = bimodal_thresh(m, bt_startbins, bt_maxbins)
    tht, mov = conditioned_theta_thresh(theta_metric, sw_metric, m, sw_thresh, mt, bt_startbins, bt_maxbins)
    dip, p = diptest(m)
    motion_metrics[name] = (m, mt, mov, tht)
    print(f"{name:14s} {mt:7.3f} {mov.mean():8.3f} {tht:7.3f} {dip:7.4f} {p:6.3f}")
print(f"(ref: theta_metric dip={diptest(theta_metric)[0]:.4f} p={diptest(theta_metric)[1]:.3f} "
      f"-> not bimodal until conditioned; NREM frac={(sw_metric>sw_thresh).mean():.3f})")


In [ ]:
# Normalised motion distributions with their trough thresholds (MOV frac + conditioned THthresh per title).
fig, ax = plt.subplots(1, len(motion_metrics), figsize=(14, 3.2), dpi=150, sharey=True)
for a, (name, (m, mt, mov, tht)) in zip(ax, motion_metrics.items()):
    a.hist(m, bins=50, density=True, alpha=0.5)
    a.axvline(mt, color="r", ls="--")
    a.set_title(f"{name}\nMOV={mov.mean():.2f}, THcond={tht:.3f}")
    a.set_xlabel("motion metric [0,1]")
ax[0].set_ylabel("density")
fig.tight_layout()


### REM-candidate diagnostic

REM should be **low SW & low muscle tone & high theta**. Here we check whether the low-SW & low-tone
overlap (REM candidates) is populated at all, how much it coincides with NREM, and whether theta is
actually elevated there. Uses the EMG (tone) metrics from the concordant block above.

In [ ]:
# REM-candidate diagnostic (EMG as tone). Uses the concordant EMG metrics (motion_metric = smooth_norm(emg_b),
# motion_thresh, th_thresh) from the concordant-thresholds cell above.
not_nrem = sw_metric < sw_thresh
low_tone = motion_metric < motion_thresh           # EMG tone below its trough (atonia)
rem_cand = not_nrem & low_tone                      # low SW & atonic -> REM candidates
rem = rem_cand & (theta_metric > th_thresh)         # + high theta

print(f"NREM frac={(~not_nrem).mean():.3f}   low-tone frac={low_tone.mean():.3f}")
print(f"of low-tone epochs, frac also NREM={(~not_nrem & low_tone).sum()/max(low_tone.sum(), 1):.3f}")
print(f"REM candidates (lowSW & lowtone)={rem_cand.mean():.3f}  ({rem_cand.sum()} epochs)")
print(f"REM (+theta>THcond={th_thresh:.3f})={rem.mean():.3f}  ({rem.sum()} epochs)")
print(f"median theta: REM-cand={np.median(theta_metric[rem_cand]):.3f} vs rest={np.median(theta_metric[~rem_cand]):.3f}")

plt.figure(figsize=(6, 4), dpi=150)
plt.hist(theta_metric[~rem_cand], bins=50, density=True, alpha=0.4, label="not REM-cand")
if rem_cand.any():
    plt.hist(theta_metric[rem_cand], bins=30, density=True, alpha=0.4, label=f"REM-cand ({rem_cand.sum()} ep)")
plt.axvline(th_thresh, color="r", ls="--", label=f"THcond={th_thresh:.3f}")
plt.xlabel("theta metric [0,1]"); plt.ylabel("density"); plt.legend()
plt.title("REM candidates: low SW & low tone")
